# 01 - Load data

This notebook loads the bundled **synthetic toy** TCR-pMHC pairs, inspects the
canonical schema, and makes a grouped train/val/test split.

Everything here runs **offline** with no heavy dependencies. To run on a real
dataset instead, flip the single commented line in the next cell to call
`load_vdjdb()` (requires network the first time, then caches).

In [ ]:
from tcr_cliff.data import load_toy, split_pairs
from tcr_cliff.data.schema import ALL_COLUMNS, REQUIRED_COLUMNS, AA_ALPHABET
from tcr_cliff.config import DataConfig

# --- OFFLINE (default): bundled synthetic toy set --------------------------
df = load_toy()

# --- REAL DATA (one-line switch): uncomment to use VDJdb instead -----------
# from tcr_cliff.data.download import load_vdjdb
# df = load_vdjdb()            # downloads + caches to ~/.cache/tcr_cliff

print(f'loaded {len(df)} pairs')
df.head()

## Canonical schema

Every loader (toy, CSV, VDJdb, McPAS, IEDB, NetTCR) normalises to the same
columns, so the rest of the pipeline is source-agnostic.

In [ ]:
print('required columns:', REQUIRED_COLUMNS)
print('all columns     :', ALL_COLUMNS)
print('AA alphabet     :', AA_ALPHABET)
print()
print('columns present :', list(df.columns))
df[['pair_id', 'cdr3b', 'peptide', 'mhc', 'binder', 'affinity']].head()

## Class balance and peptide diversity

The toy set is constructed so that single-residue *neighbours* of a peptide can
flip the binding label - the activity-cliff phenomenon this package targets.

In [ ]:
print('binder balance:')
print(df['binder'].value_counts())
print()
print('n unique peptides:', df['peptide'].nunique())
print('n unique CDR3b   :', df['cdr3b'].nunique())
print('peptide lengths  :', sorted(df['peptide'].str.len().unique()))

## Grouped train/val/test split

`split_pairs` honours an existing `split` column when present; otherwise it does a
**grouped** split (default on `peptide`) so that the two halves of a cliff pair -
and the same peptide - never straddle train and test. That prevents leaking the
very signal the benchmark measures.

In [ ]:
# Random grouped split on peptide (ignore the bundled `split` column for the demo).
cfg = DataConfig(group_split_on='peptide', split_column='__none__')
parts = split_pairs(df, cfg, seed=0)
for name, part in parts.items():
    print(f'{name:6s}: {len(part):4d} rows | binders={int(part["binder"].sum())}')

# Sanity check: no peptide appears in both train and test (grouped split).
overlap = set(parts['train']['peptide']) & set(parts['test']['peptide'])
print('peptide overlap train/test:', len(overlap))

### Next
Continue to **02_embeddings** to turn these sequences into feature vectors.